#### Verify Paths and Confirm Right Env ###
Run the next code block to to make paths robust and confirm we’re in the right env.

In [ ]:
# --- environment + paths sanity check ---
from pathlib import Path
import sys, platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kernel env path:", sys.executable)

# project layout: PS4/ {Input, Output, ...}
nb_dir = Path.cwd()                    # VS Code's notebook cwd is the file's folder
project_root = nb_dir.parents[1]       # PS4/
input_dir = project_root / "Input"     # PS4/Input
output_dir = project_root / "Output"   # PS4/Output

print("Notebook dir:", nb_dir)
print("Input dir exists:", input_dir.exists())
print("Output dir exists:", output_dir.exists())

#### Import Stack + Data Sanity Check ###
Run this next chunk so we can verify that we load the data properly.

In [ ]:
# core stack
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("statsmodels:", sm.__version__)

# pandas display tweaks (optional)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# load data (relative to PS4/Input/)
csv_path = (input_dir / "welfare.csv")
assert csv_path.exists(), f"Missing file: {csv_path}"
ssp = pd.read_csv(csv_path)

# quick peek
print("Rows, Cols:", ssp.shape)
print(sorted(ssp.columns.tolist())[:12], "...")

# the “treatment” var check from the template
print(ssp["treatment"].value_counts(dropna=False))

# Question 2: Self Sufficiency Program (see ssp.ipynb) #

Note: Write your code in the code cells, and your responses in markdown. Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Friday, 10/24**.  Upload this notebook AND a HTML/PDF file to Canvas. Please make sure you have an AI disclaimer that indicates where/how you use AI tool. 

<!-- Helpful tips on Markdown: - https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html# -->

In [ ]:
# --- Environment & Paths Setup ---
from pathlib import Path
import sys, platform

# Confirm environment and platform
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kernel path:", sys.executable)

# Define the notebook directory
nb_dir = Path.cwd()                   # Should be PS4/Input/
project_root = nb_dir.parents[0]      # PS4/
input_dir = nb_dir                    # Input/ (same folder)
output_dir = project_root / "Output"  # PS4/Output/

print("\nNotebook dir:", nb_dir)
print("Input dir:", input_dir)
print("Output dir:", output_dir)
print("Input dir exists:", input_dir.exists())
print("Output dir exists:", output_dir.exists())

In [ ]:
# --- Import Stack + Data Sanity Check ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import matplotlib.pyplot as plt

# Verify library versions
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("statsmodels:", sm.__version__)

# Load data
csv_path = input_dir / "welfare.csv"
assert csv_path.exists(), f"Missing file: {csv_path}"

ssp = pd.read_csv(csv_path)

# Quick inspection
print("Rows, Cols:", ssp.shape)
print(sorted(ssp.columns.tolist())[:12], "...")
print("\nTreatment variable check:")
print(ssp["treatment"].value_counts(dropna=False))

## Self Sufficiency Project 
- Background from Card and Hyslop (2005): 
    - In the Self Sufficiency Project (SSP), members of a randomly assigned treatment group could receive a subsidy for full-time work. The subsidy was available for 3 years, but only to people who began working full time within 12 months of random assignment.  
    - SSP provides two incentives: (1) find a job and leave welfare within a year, and (2) choose work over welfare in the longer term. 

- This question is designed to apply the IV-LATE framework to estimate the causal effects of SSP on leaving welfare, and characterize the compliers who would not have found a full-time job in the absence of the SSP subsidy. 

- The data set welfare.csv contains 5,480 observations for people in the SSP experiment. See PS4.pdf for variable definitions. 

In [ ]:
# Load dataset (using pandas "pd")
ssp = pd.read_csv("welfare.csv") # add your own directory if necessary
print(ssp.columns)
print(ssp['treatment'].value_counts(dropna=False)) 

In [ ]:
print(ssp['treatment'].value_counts()) # 1 if assigned to the treatment group (receiving a subsidy)
print(ssp['welfare15'].value_counts()) # 1 if on welfare at t=15. 

In [ ]:
# Note Missing Values: 
print(ssp[['ft15','ft20','ft24','ft48','treatment','welfare15','welfare20','welfare24','welfare48']].isnull().sum())
# input data in IV2SLS needs to be nonmissing in y/x1/x2/z 
# e.g., for t=24, input  data = ssp.loc[~ssp['ft15'].isnull()]

Note ft20/ft24/ft48 are missing in some rows. We need estimate the first-stage, the reduced form, and the 2SLS on the same sample, conditional on $FT_i(t)$ is nonmissing at a given $t$. For example, at $t=20$, estimate the regressions on the sample below: 

In [ ]:
t=20
print(ssp.loc[~ssp[f'ft{t}'].isnull()].head(2))  

### 2.1 First Stage
Estimate first stage models for the probability of working FT in months 15, 20, 24, 48, using treatment as the instrumental variable. $$FT_{i}(t)=\pi_{0}+\pi_{1}\text{treatment}_{i}+\epsilon_{i},\,\text{ for }t=15,20,24,28$$

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

ts = [15, 20, 24, 48]     # months
rows = []

for t in ts:
    # keep rows where ft{t} is observed, and rename to a generic 'ft'
    df = ssp.loc[~ssp[f'ft{t}'].isnull(), ['treatment', f'ft{t}']].rename(columns={f'ft{t}':'ft'})

    # OLS with robust SE (this is the first stage)
    res = smf.ols('ft ~ treatment', data=df).fit(cov_type='HC1')

    # collect pieces
    pi0 = res.params['Intercept']
    pi1 = res.params['treatment']
    se1 = res.bse['treatment']
    tval = res.tvalues['treatment']
    fstat = float(tval**2)              # single regressor => F = t^2
    n = int(res.nobs)

    # quick means check (difference-in-means identity)
    m = df.groupby('treatment')['ft'].mean()
    p0 = float(m.get(0, float('nan')))
    p1 = float(m.get(1, float('nan')))

    rows.append({
        't'      : t,
        'pi0'    : pi0,                 # intercept
        'pi1'    : pi1,                 # first-stage effect
        'SE(pi1)': se1,
        't(pi1)' : tval,
        'F(pi1)' : fstat,
        'N'      : n,
        'Pr[FT=1|Z=0]' : p0,            # should equal pi0
        'Pr[FT=1|Z=1]' : p1,            # p1 - p0 should equal pi1
    })

first_stage = pd.DataFrame(rows).set_index('t')
first_stage

for t in ts:
    df = ssp.loc[~ssp[f'ft{t}'].isnull(), ['treatment', f'ft{t}']].rename(columns={f'ft{t}':'ft'})
    res = smf.ols('ft ~ treatment', data=df).fit(cov_type='HC1')
    print(f'\n=== First stage for month t={t} ===')
    print(res.summary())

__Interpretation of $\pi_0$ and $\pi_1$__:

Here $\pi_0$ represents the baseline probability of working full-time among control group participants, while $\pi_1$ captures the increase in that probability caused by being assigned to the SSP treatment (receiving a subsidy offer).

The estimates show that π₀ rises from 0.15 to 0.23 between 15 and 48 months, reflecting gradual recovery in the control group. $\pi_0$ declines from 0.137 at 15 months to 0.051 at 48 months, indicating that the effect of SSP assignment on full-time work was strongest early on and diminished over time as the program’s incentive expired.

These results confirm that treatment assignment is a strong instrument (F > 10 in all cases) and is positively associated with working full-time.

### 2.2 Reduced Form: Welfare recipience on random assignment. 
Estimate reduced form models for the probability of being on welfare in months 15, 20, 24, 48, using treatment as the instrumental variable. $$\text{Welfare}_{i}(t)=\delta_{0}+\delta_{1}\text{treatment}_{i}+\nu_{i},\,\text{ for }t=15,20,24,28$$ where $\text{Welfare}_{i}(t)=1$ if person $i$ is on welfare $t$ months since random assignment. 

In [ ]:
# Hint: make sure to estimate the regression conditional on "ft" is nonmissing at each given t; see the hint for first stage. 

# ---- Reduced form: Welfare_t = δ0 + δ1 * treatment + ν  (t = 15,20,24,48) ----
import pandas as pd
import statsmodels.formula.api as smf

ts = [15, 20, 24, 48]
rows = []

for t in ts:
    # same sample rule as the handout: keep rows where FT_t is observed
    df = ssp.loc[~ssp[f'ft{t}'].isnull(), ['treatment', f'welfare{t}']].rename(columns={f'welfare{t}': 'w'})
    res = smf.ols('w ~ treatment', data=df).fit(cov_type='HC1')
    
    delta0 = res.params['Intercept']
    delta1 = res.params['treatment']
    se1    = res.bse['treatment']
    t1     = res.tvalues['treatment']
    F1     = float(t1**2)  # same as single-regressor F
    N      = int(df.shape[0])

    # group means (nice check that δ0 = Pr[W=1|Z=0] and δ0+δ1 = Pr[W=1|Z=1])
    p_w_z0 = df.loc[df.treatment==0, 'w'].mean()
    p_w_z1 = df.loc[df.treatment==1, 'w'].mean()

    rows.append([t, delta0, delta1, se1, t1, F1, N, p_w_z0, p_w_z1])

rf_tbl = pd.DataFrame(rows, columns=[
    't','delta0','delta1','SE(delta1)','t(delta1)','F(delta1)','N',
    'Pr[W=1|Z=0]','Pr[W=1|Z=1]'
]).set_index('t').round(6)

rf_tbl

__Reduced-form interpretation__

The estimates show that random assignment to the Self-Sufficiency Program (SSP) significantly reduced welfare receipt. At 15 months, about 81 % of the control group remained on welfare, compared with 67 % of those offered the SSP subsidy—a 14 percentage-point decline (p < 0.001). The magnitude of this effect gradually declined to 12 p.p. at 20 months, 11 p.p. at 24 months, and 4 p.p. at 48 months. This pattern is consistent with the program’s temporary three-year subsidy period and suggests that SSP assignment led participants to exit welfare sooner, though some differences dissipated once the subsidy ended.

In this regression, δ₀ represents the average probability of welfare participation among controls, while δ₁ captures the intent-to-treat effect of assignment on welfare status.

### 2.3 Second Stage (2SLS) 
Estimate 2SLS models of the equation: $$\text{Welfare}_{i}(t)=\beta_{0}+\beta_{1}FT_{i}(t)+u_{i},\,\text{ for }t=15,20,24,28$$ using treatment as an instrument for $FT_i(t)$. Verify that the 2SLS estimate in each case is the ratio of the reduced form and first stage coefficients. Hint: using linearmodel IV2SLS. 

In [ ]:
# Example if using IV2SLS: 
# from linearmodels.iv import IV2SLS 
# model = IV2SLS.from_formula(
#     'y ~ 1 + x1 + [x2 ~ z1 + z2]',  # x2 is endogenous, z1,z2 are instruments
#     data=your_dataframe  # note the variables needs to nonmissing in the data
# )
# results = model.fit()
# print(results.summary)
# print(results.params)
print(ssp[['ft15','ft20','ft24','ft48','treatment','welfare15','welfare20','welfare24','welfare48']].isnull().sum())
# input data in IV2SLS needs to be nonmissing in y/x1/x2/z 
# e.g., for t=24, input  data = ssp.loc[~ssp['ft15'].isnull()]

In [ ]:
from linearmodels.iv import IV2SLS
import statsmodels.formula.api as smf
import pandas as pd

ts = [15, 20, 24, 48]
rows = []

for t in ts:
    # restrict to rows where ft(t) is observed
    df = (
        ssp.loc[~ssp[f'ft{t}'].isnull(), ['treatment', f'ft{t}', f'welfare{t}']]
           .rename(columns={f'ft{t}': 'ft', f'welfare{t}': 'w'})
    )

    # 2SLS:  w ~ 1 + [ft ~ treatment]
    iv = IV2SLS.from_formula('w ~ 1 + [ft ~ treatment]', data=df).fit(cov_type='robust')

    # reduced form (w on treatment) and first stage (ft on treatment), HC1 SEs
    rf = smf.ols('w ~ treatment', data=df).fit(cov_type='HC1')
    fs = smf.ols('ft ~ treatment', data=df).fit(cov_type='HC1')

    rows.append({
        't': t,
        'beta0': iv.params['Intercept'],
        'beta1': iv.params['ft'],
        'SE(beta1)': iv.std_errors['ft'],
        't(beta1)': iv.tstats['ft'],
        'N': iv.nobs,
        'delta1 (RF)': rf.params['treatment'],
        'pi1 (FS)': fs.params['treatment'],
        'delta1/pi1 check': rf.params['treatment'] / fs.params['treatment'],
    })

tbl_2sls = pd.DataFrame(rows).set_index('t').round(6)
tbl_2sls

### 2.4 Characteristics of Compliers, Always Takers, and Never Takers
Find the mean characteristics of the compliers in month $t=15$: i.e., means of the variables imm, hsgrad, agelt25, age35p, working_at_baseline, anykidsu6, and nevermarried. Compare these to the characteristics of always takers and never takers in month 15. Hint: define outcome $Y_{i}=X_{i}\times FT_{i}(15)$. 

In [ ]:
# variables to summarize
X = ['imm', 'hsgrad', 'agelt25', 'age35p', 'working_at_baseline', 'anykidsu6', 'nevermarried']

# for t = 15 only
t = 15

# define the relevant indicators
ft = f'ft{t}'
Z = 'treatment'

# identify groups based on the logic of potential outcomes
always_takers = ssp.loc[(ssp[ft] == 1) & (ssp[Z] == 0), X].mean()
never_takers  = ssp.loc[(ssp[ft] == 0) & (ssp[Z] == 1), X].mean()

# estimate compliers by the Wald formula weights
# E[X|Z=1] - E[X|Z=0] divided by P(FT=1|Z=1) - P(FT=1|Z=0)
E_X1 = ssp.loc[ssp[Z] == 1, X].mean()
E_X0 = ssp.loc[ssp[Z] == 0, X].mean()
P1 = ssp.loc[ssp[Z] == 1, ft].mean()
P0 = ssp.loc[ssp[Z] == 0, ft].mean()
share_compliers = P1 - P0
compliers = (E_X1 - E_X0) / share_compliers

# combine and display results
tbl_chars = pd.DataFrame({
    'Compliers': compliers,
    'Always Takers': always_takers,
    'Never Takers': never_takers
}).round(3)

tbl_chars

### 2.4 Characteristics of Compliers, Always Takers, and Never Takers

**Goal.** For $t=15$, report the mean characteristics of **compliers** and compare them with **always takers** and **never takers** for  
$\{\texttt{imm}, \texttt{hsgrad}, \texttt{agelt25}, \texttt{age35p}, \texttt{working\_at\_baseline}, \texttt{anykidsu6}, \texttt{nevermarried}\}$.

---

**Method (what we computed).**  
Let $Z$ be assignment to treatment and $FT(15)$ the indicator for working full-time at 15 months.

- **Always takers:** $E[X\mid FT=1,\,Z=0]$
- **Never takers:** $E[X\mid FT=0,\,Z=1]$
- **Compliers (Wald / LATE weights):**
$
E[X\mid \text{complier}]
=\frac{E[X\mid Z=1]-E[X\mid Z=0]}{P(FT=1\mid Z=1)-P(FT=1\mid Z=0)} .
$

Using the sample shares $P(FT=1\mid Z=0)=0.148$ and $P(FT=1\mid Z=1)=0.285$, the estimates are:

| Characteristic | Compliers | Always Takers | Never Takers |
|---|---:|---:|---:|
| imm | -0.026 | 0.078 | 0.139 |
| hsgrad | 0.117 | 0.615 | 0.414 |
| agelt25 | -0.058 | 0.172 | 0.170 |
| age35p | 0.033 | 0.298 | 0.327 |
| working_at_baseline | -0.048 | 0.468 | 0.122 |
| anykidsu6 | -0.145 | 0.502 | 0.520 |
| nevermarried | -0.011 | 0.485 | 0.483 |

---

**Interpretation.**

- **Compliers** look **intermediate** between always and never takers—consistent with the IV-LATE idea that the policy moves a marginal group. They are **less likely to be immigrants** and to have **young children**, and they were **less likely to be working at baseline** than always takers. Their education ($\texttt{hsgrad}$) is below always takers but above never takers.
- **Always takers** are the most advantaged: higher education, older ($\texttt{age35p}$), and much more likely to have been working at baseline—i.e., they would work regardless of assignment.
- **Never takers** are more constrained (lower $\texttt{hsgrad}$, low baseline work, more with young kids) and do not respond to the subsidy.

Overall, the SSP subsidy appears to induce full-time work primarily among **moderately disadvantaged individuals**—those who are not as attached to the labor market as always takers but face fewer barriers than never takers.